# Regression_LLM — California Housing Price Prediction
### CSC-114 Artificial Intelligence I · Module 4 · Scalar Regression

**Companion to:** Chollet & Watson, *Deep Learning with Python*, 3rd ed., Ch. 4 — Predicting House Prices.

This notebook builds a scalar regression model using the California Housing dataset.
The model predicts the **median home price** of a district given 8 numeric features.

**Backend:** TensorFlow (explicitly set below — JAX is not used here.)

---

### What this notebook does, step by step

1. Set the backend to TensorFlow
2. Load the California Housing dataset
3. Normalize features and scale targets
4. Build the regression model
5. Validate using K-fold cross-validation
6. Find the right number of epochs from the validation curve
7. Train the final model and evaluate on the test set
8. Generate predictions on new data

## Step 1 — Set the backend to TensorFlow

The backend must be set **before** importing Keras.
This notebook uses TensorFlow. If you previously set `KERAS_BACKEND` to `jax`
or `torch` in another notebook, this cell overrides it for this session.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Force TensorFlow — not JAX, not PyTorch

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

print("Keras version :", keras.__version__)
print("Backend       :", keras.backend.backend())
# CHECKPOINT: backend should print 'tensorflow', not 'jax' or 'torch'.

## Step 2 — Load the California Housing dataset

We use the **small** version: 480 training districts and 120 test districts.
Each district has **8 numeric features**. The target is the median home price in dollars.

In [ ]:
from keras.datasets import california_housing

(train_data, train_targets), (test_data, test_targets) = (
    california_housing.load_data(version="small")
)

print("Training samples :", train_data.shape)   # (480, 8)
print("Test samples     :", test_data.shape)     # (120, 8)
print("Sample targets   :", train_targets[:5])   # dollar prices
# CHECKPOINT: shape should be (480, 8) for training and (120, 8) for test.

## Step 3 — Normalize the features

The 8 features are on wildly different scales (longitude vs. population vs. income).
Feeding them straight in makes learning very difficult.

**Feature normalization:** subtract the mean and divide by the standard deviation
for each feature. After this, every feature is centered around 0.

**Critical rule:** compute mean and std from the **training data only**.
Apply those same numbers to normalize the test data.
Never compute anything from the test data — even for normalization.

In [ ]:
# Compute mean and std from training data only
mean = train_data.mean(axis=0)   # one mean per feature (column)
std  = train_data.std(axis=0)    # one std  per feature (column)

# Normalize both sets using TRAINING statistics
x_train = (train_data - mean) / std
x_test  = (test_data  - mean) / std

# Scale targets: divide by 100,000 so predictions stay in a small range.
# Multiply predictions back by 100,000 to recover dollar values.
y_train = train_targets / 100000
y_test  = test_targets  / 100000

print("x_train mean (should be ~0):", x_train.mean(axis=0).round(3))
print("x_train std  (should be ~1):", x_train.std(axis=0).round(3))
print("y_train range: %.2f to %.2f" % (y_train.min(), y_train.max()))
# CHECKPOINT: means near 0, stds near 1, y range roughly 0.6 to 5.0.

## Step 4 — Define the model

Three key differences from the classification models:

- **Final layer: `Dense(1)` with NO activation** — output is unconstrained, any value.
- **Loss: `mean_squared_error`** — squares each miss, so large errors hurt far more.
- **Metric: `mean_absolute_error`** — average dollars off; MAE of 0.31 = ~$31,000 off.

Wrapped in a function because K-fold validation needs a fresh model each fold.

In [ ]:
def get_model():
    model = keras.Sequential([
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),   # No activation — regression output, any value
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
        metrics=["mean_absolute_error"],
    )
    return model

# Quick check: build one model and print its summary
get_model().summary()

## Step 5 — K-fold cross-validation (50 epochs)

With only 480 training samples, a single validation split is unreliable —
the score shifts dramatically depending on which samples you picked.

**K-fold:** split the data into 4 equal folds. Train 4 separate models,
each time using a different fold as the validation set.
Average the 4 scores — that average is the reliable number.

```
Fold 1:  [VAL] [train] [train] [train]
Fold 2:  [train] [VAL] [train] [train]
Fold 3:  [train] [train] [VAL] [train]
Fold 4:  [train] [train] [train] [VAL]
```

In [ ]:
k = 4
num_val_samples = len(x_train) // k   # 480 // 4 = 120 samples per fold
num_epochs = 50
all_scores = []

for i in range(k):
    print(f"Processing fold #{i + 1}")

    # Validation slice for this fold
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]

    # Everything else becomes the training slice
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )

    model = get_model()   # Fresh untrained model each fold
    model.fit(
        fold_x_train, fold_y_train,
        epochs=num_epochs,
        batch_size=16,
        verbose=0,   # Silent — 4 folds x 50 epochs = too many lines
    )
    scores = model.evaluate(fold_x_val, fold_y_val, verbose=0)
    val_loss, val_mae = scores
    all_scores.append(val_mae)

print("\nMAE per fold :", [round(v, 3) for v in all_scores])
print("Average MAE  :", round(np.mean(all_scores), 3))
print("Avg dollars off: $%.0f" % (np.mean(all_scores) * 100000))
# CHECKPOINT: individual scores will vary; average should be around 0.29-0.32.

## Step 6 — K-fold over 200 epochs to find the overfitting point

50 epochs may not be enough. We run K-fold again for 200 epochs,
saving the validation MAE at every epoch across all 4 folds.
Then we average per epoch to find where validation MAE stops improving.

In [ ]:
k = 4
num_val_samples = len(x_train) // k
num_epochs = 200
all_mae_histories = []

for i in range(k):
    print(f"Processing fold #{i + 1}")

    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )

    model = get_model()
    history = model.fit(
        fold_x_train, fold_y_train,
        validation_data=(fold_x_val, fold_y_val),
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
    )
    mae_history = history.history["val_mean_absolute_error"]
    all_mae_histories.append(mae_history)

# Average MAE at each epoch across all 4 folds
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)
]

print("Done. Plotting validation MAE curve...")

## Step 7 — Plot the validation MAE curve

Look for where the curve **stops improving** — that's the safe epoch count.
The first 10 epochs are dropped from the plot because early MAE is
on a different scale and makes the rest hard to read.

In [ ]:
# Full curve
epochs_full = range(1, len(average_mae_history) + 1)
plt.plot(epochs_full, average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.title("K-fold Validation MAE — all 200 epochs")
plt.show()

# Zoomed: drop the first 10 noisy epochs
truncated = average_mae_history[10:]
epochs_zoom = range(10, len(truncated) + 10)
plt.plot(epochs_zoom, truncated)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.title("K-fold Validation MAE — epochs 10 onward (zoomed)")
plt.show()

# CHECKPOINT: the curve should flatten and then start rising around epoch 120-140.
# That rising point is where overfitting begins. Use 130 as the safe epoch count.

## Step 8 — Train the final model and evaluate

Now that we know the right epoch count (~130), we:
1. Train a fresh model on **all** training data (no validation split needed)
2. Evaluate **once** on the sealed test set — the honest final grade

Multiply MAE by 100,000 to convert back to dollars.

In [ ]:
model = get_model()
model.fit(
    x_train, y_train,
    epochs=130,
    batch_size=16,
    verbose=0,
)

test_mse, test_mae = model.evaluate(x_test, y_test, verbose=0)

print("Test MSE        : %.4f" % test_mse)
print("Test MAE        : %.4f" % test_mae)
print("Dollars off avg : $%.0f" % (test_mae * 100000))
# CHECKPOINT: MAE should be around 0.28-0.33, meaning ~$28k-$33k off on average.
# Your exact number wobbles a little each run due to random weight initialization.

## Step 9 — Generate predictions on new data

`predict()` returns a single number per district — the predicted median home price
in units of $100,000. Multiply by 100,000 to get dollar values.

Compare to binary classification (one 0-1 probability) and
multiclass classification (46 probabilities summing to 1).
Regression just returns **one unconstrained number**.

In [ ]:
predictions = model.predict(x_test, verbose=0)

print("First 5 predictions vs actual prices:")
print(f"{'Predicted':>12}  {'Actual':>12}  {'Off by':>12}")
print("-" * 42)
for i in range(5):
    pred   = predictions[i][0] * 100000
    actual = y_test[i]         * 100000
    diff   = abs(pred - actual)
    print(f"${pred:>10,.0f}  ${actual:>10,.0f}  ${diff:>10,.0f}")

## Wrap-up — what to take from this notebook

| Concept | Regression choice | Why |
|---|---|---|
| Final layer | `Dense(1)` — no activation | Output must be any value |
| Loss | `mean_squared_error` | Penalizes large misses heavily |
| Metric | `mean_absolute_error` | Readable as dollars off |
| Preprocessing | Normalize each feature | Different scales break learning |
| Validation | K-fold (k=4) | Too little data for a single split |
| Backend | TensorFlow | Explicitly set — not JAX |

---
*Source: Chollet & Watson, Deep Learning with Python, 3rd Edition (Manning), Chapter 4.*
*Backend explicitly set to TensorFlow per assignment requirements.*